# Anticipez les besoins en consommation de bâtiments — Modélisation

Suite de `01_analyse_exploratoire.ipynb` : on repart du jeu nettoyé (bâtiments non résidentiels, relevés conformes et cohérents).

# Modélisation 

### Import des modules 

In [ ]:
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

#Selection
from sklearn.model_selection import (
    train_test_split,
    GridSearchCV,
    cross_validate,
)
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.inspection import permutation_importance

#Preprocess
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler

#Modèles
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor

pd.set_option("display.max_columns", 60)
sns.set_theme(style="whitegrid")

### Chargement du jeu nettoyé

In [ ]:
df = pd.read_csv("../data/building_consumption_clean.csv", index_col="OSEBuildingID")
cible = "SiteEnergyUse(kBtu)"
df.shape

### Feature Engineering

A réaliser : Enrichir le jeu de données actuel avec de nouvelles features issues de celles existantes.

**Règle suivie pour chaque feature** : elle doit pouvoir être calculée pour un bâtiment **jamais mesuré**.
On n'utilise donc jamais la **valeur** d'une colonne issue du relevé (consommations, intensités, émissions, ENERGY STAR Score).

Les features couvrent cinq catégories d'informations : **temporalité**, **structure**, **usages**, **sources d'énergie** et **localisation**.
Toutes sont calculées ligne par ligne, à partir du bâtiment lui-même, ou avec des règles fixées à l'avance (seuils, regroupements métier) :
aucune n'utilise une statistique calculée sur l'ensemble du jeu, ce qui évite une fuite entre futurs jeux d'entraînement et de test.

#### 1. Temporalité

- `BuildingAge` : âge du bâtiment au moment du relevé (2016). Plus parlant que l'année et directement interprétable.
- `PeriodeConstruction` : tranches correspondant à de grandes époques constructives — avant 1946 (bâti ancien), 1946-1979 (après-guerre,
  avant les premières réglementations énergétiques consécutives aux chocs pétroliers), 1980-2000, après 2000 (normes plus récentes).
  Les tranches permettent à un modèle linéaire de capter un effet **non linéaire** de l'âge.

In [ ]:
ANNEE_RELEVE = 2016
df["BuildingAge"] = ANNEE_RELEVE - df["YearBuilt"]
df["PeriodeConstruction"] = pd.cut(
    df["YearBuilt"],
    bins=[1899, 1945, 1979, 2000, ANNEE_RELEVE],
    labels=["Avant 1946", "1946-1979", "1980-2000", "Après 2000"],
)
df["PeriodeConstruction"].value_counts().sort_index()

#### 2. Structure du bâtiment

- `LogSurfaceTotale` : log de la surface totale. L'analyse exploratoire a montré une relation **linéaire en log-log** entre surface et consommation.
- `PartParking` : part de la surface totale occupée par du parking, peu consommateur (éclairage, ventilation) comparé aux bureaux ou commerces.
- `SurfaceParEtage` : surface bâtie moyenne par étage — distingue un entrepôt de plain-pied très étendu d'une tour compacte.
- `SurfaceParBatiment` : surface bâtie moyenne par bâtiment de la propriété.
- `MultiBatiments` : 1 si la propriété compte plusieurs bâtiments (campus, hôpitaux…).

Pas de risque de division par zéro : l'analyse exploratoire a supprimé les bâtiments à 0 étage et corrigé les propriétés à 0 bâtiment.

In [ ]:
df["LogSurfaceTotale"] = np.log(df["PropertyGFATotal"])
df["PartParking"] = df["PropertyGFAParking"] / df["PropertyGFATotal"]
df["SurfaceParEtage"] = df["PropertyGFABuilding(s)"] / df["NumberofFloors"]
df["SurfaceParBatiment"] = df["PropertyGFABuilding(s)"] / df["NumberofBuildings"]
df["MultiBatiments"] = (df["NumberofBuildings"] > 1).astype(int)

df[["LogSurfaceTotale", "PartParking", "SurfaceParEtage", "SurfaceParBatiment", "MultiBatiments"]].describe().T.round(2)

#### 3. Usages du bâtiment

**Nombre d'usages.** `ListOfAllPropertyUseTypes` liste les usages séparés par des virgules.
⚠️ Piège : certains libellés contiennent eux-mêmes des virgules entre parenthèses, par exemple
`Repair Services (Vehicle, Shoe, Locksmith, etc)`. Un simple `split(",")` compterait 4 usages au lieu d'un :
on retire d'abord le contenu des parenthèses.

In [ ]:
def lister_usages(texte):
    """Renvoie la liste des usages d'un bâtiment à partir de ListOfAllPropertyUseTypes."""
    if pd.isna(texte):
        return []
    texte = re.sub(r"\([^)]*\)", "", texte)  # retire « (Vehicle, Shoe, …) » qui contient des virgules
    return [usage.strip() for usage in texte.split(",") if usage.strip()]

usages = df["ListOfAllPropertyUseTypes"].apply(lister_usages)

# Vérification du piège : sans retirer les parenthèses, on compterait trop d'usages
naif = df["ListOfAllPropertyUseTypes"].fillna("").str.split(",").str.len()
print("Bâtiments dont le comptage naïf serait faux :", (naif != usages.str.len()).sum())
print("Liste des usages absente :", df["ListOfAllPropertyUseTypes"].isna().sum())

La liste n'est absente pour aucun bâtiment du jeu nettoyé ; par sécurité (bâtiment futur sans liste), on borne tout de même `NbUsages` à 1 :
un bâtiment a au moins un usage, son `PrimaryPropertyType`.

In [ ]:
df["NbUsages"] = usages.str.len().clip(lower=1)
df["MultiUsage"] = (df["NbUsages"] > 1).astype(int)
df["NbUsages"].value_counts().sort_index()

**Proportions de surface par usage.** Indiquent si le bâtiment est mono-usage (part principale proche de 1) ou mixte.
- Les 2ᵉ et 3ᵉ usages manquent **par construction** quand le bâtiment n'en a pas : leur surface vaut donc **0**.
- Les parts sont bornées à 1 : l'analyse exploratoire a montré que la surface d'un usage peut légèrement dépasser la surface totale (écart ≤ 10 % conservé).
- `PartUsagePrincipal` reste manquante pour les quelques bâtiments sans usage principal renseigné : elle sera imputée lors de la préparation.

In [ ]:
df["PartUsagePrincipal"] = (df["LargestPropertyUseTypeGFA"] / df["PropertyGFATotal"]).clip(upper=1)
df["PartUsage2"] = (df["SecondLargestPropertyUseTypeGFA"].fillna(0) / df["PropertyGFATotal"]).clip(upper=1)
df["PartUsage3"] = (df["ThirdLargestPropertyUseTypeGFA"].fillna(0) / df["PropertyGFATotal"]).clip(upper=1)
df[["PartUsagePrincipal", "PartUsage2", "PartUsage3"]].describe().T.round(2)

**Présence d'usages très énergivores.** Un bâtiment de bureaux qui abrite un restaurant, un data center ou une piscine consommera plus
qu'un bureau classique. On crée une feature binaire par grande famille d'usage intensif, à partir de **tous** les usages du bâtiment
(pas seulement le principal).

In [ ]:
usages_intensifs = {
    "UsageRestauration": ["Restaurant", "Fast Food Restaurant", "Food Service", "Other - Restaurant/Bar", "Bar/Nightclub"],
    "UsageAlimentaire": ["Supermarket/Grocery Store", "Food Sales", "Convenience Store without Gas Station", "Wholesale Club/Supercenter"],
    "UsageSante": ["Hospital", "Other/Specialty Hospital", "Urgent Care/Clinic/Other Outpatient", "Laboratory"],
    "UsageDataCenter": ["Data Center"],
    "UsagePiscineSport": ["Swimming Pool", "Fitness Center/Health Club/Gym"],
    "UsageFroid": ["Refrigerated Warehouse"],
    "UsageParking": ["Parking"],
}
for colonne, liste in usages_intensifs.items():
    df[colonne] = usages.apply(lambda u: int(any(usage in liste for usage in u)))

df[list(usages_intensifs)].sum().to_frame("Nb bâtiments")

**Compression d'une catégorie à haute cardinalité.** `PrimaryPropertyType` compte une vingtaine de modalités, dont certaines
avec une dizaine de bâtiments seulement (hôpitaux, laboratoires) : trop peu pour qu'un modèle apprenne un effet fiable.
On les regroupe en **familles d'usage** proches en termes de consommation (cf. intensités vues en analyse exploratoire).
Le regroupement est une **règle métier fixée à l'avance**, pas un seuil de fréquence calculé sur les données.

De même, on privilégiera `Neighborhood` (13 quartiers) à `ZipCode` (près de 50 codes postaux) et `PrimaryPropertyType` / `FamilleUsage`
à `LargestPropertyUseType` (plus de 50 modalités, quasi identique au type principal) lors de la préparation des features.

In [ ]:
familles = {
    "Bureau": ["Large Office", "Small- and Mid-Sized Office", "Office"],
    "Santé": ["Hospital", "Medical Office", "Laboratory", "Senior Care Community"],
    "Alimentation": ["Supermarket / Grocery Store", "Restaurant"],
    "Commerce": ["Retail Store"],
    "Logistique": ["Warehouse", "Distribution Center", "Refrigerated Warehouse", "Self-Storage Facility"],
    "Enseignement": ["K-12 School", "University"],
    "Hébergement": ["Hotel", "Residence Hall"],
    "Culte": ["Worship Facility"],
    "Mixte / Autre": ["Mixed Use Property", "Other"],
}
type_vers_famille = {t: famille for famille, types in familles.items() for t in types}
df["FamilleUsage"] = df["PrimaryPropertyType"].map(type_vers_famille)

assert df["FamilleUsage"].notna().all(), "Un type de bâtiment n'a pas de famille"
df["FamilleUsage"].value_counts()

#### 4. Sources d'énergie

Les **quantités** de gaz, d'électricité ou de vapeur ne peuvent pas être utilisées : elles composent directement la cible (fuite de données).
En revanche, **savoir si le bâtiment est raccordé** au gaz ou au réseau de vapeur est une information **structurelle** : elle dépend
des équipements (chaudière gaz, raccordement au réseau urbain), connus sans relevé de consommation, et pas de l'intensité de la consommation.
- `UtiliseGaz` : 1 si le bâtiment consomme du gaz naturel.
- `UtiliseVapeur` : 1 si le bâtiment est raccordé au réseau de vapeur.

Pas de feature pour l'électricité : la quasi-totalité des bâtiments l'utilisent, la feature serait constante.

In [ ]:
df["UtiliseGaz"] = (df["NaturalGas(kBtu)"] > 0).astype(int)
df["UtiliseVapeur"] = (df["SteamUse(kBtu)"] > 0).astype(int)
print("Utilisent l'électricité :", (df["Electricity(kBtu)"] > 0).mean().round(3))
df[["UtiliseGaz", "UtiliseVapeur"]].mean().round(2).to_frame("Part des bâtiments")

#### 5. Localisation

- `DistanceCentreKm` : distance (en km, formule de haversine) au centre-ville de Seattle (47,6062 N ; 122,3321 O).
  Elle résume en une seule valeur continue la position du bâtiment, là où latitude et longitude séparées
  sont difficiles à exploiter pour un modèle linéaire.

In [ ]:
CENTRE_SEATTLE = (47.6062, -122.3321)

def distance_km(lat, lon, lat0=CENTRE_SEATTLE[0], lon0=CENTRE_SEATTLE[1]):
    """Distance à vol d'oiseau (formule de haversine) entre deux points GPS, en km."""
    lat, lon, lat0, lon0 = map(np.radians, (lat, lon, lat0, lon0))
    a = np.sin((lat - lat0) / 2) ** 2 + np.cos(lat) * np.cos(lat0) * np.sin((lon - lon0) / 2) ** 2
    return 2 * 6371 * np.arcsin(np.sqrt(a))

df["DistanceCentreKm"] = distance_km(df["Latitude"], df["Longitude"])
df.groupby("Neighborhood")["DistanceCentreKm"].median().sort_values().round(1)

#### Récapitulatif et contrôles

In [ ]:
nouvelles_features = [
    "BuildingAge", "PeriodeConstruction",
    "LogSurfaceTotale", "PartParking", "SurfaceParEtage", "SurfaceParBatiment", "MultiBatiments",
    "NbUsages", "MultiUsage", "PartUsagePrincipal", "PartUsage2", "PartUsage3", *usages_intensifs, "FamilleUsage",
    "UtiliseGaz", "UtiliseVapeur",
    "DistanceCentreKm",
]
numeriques = df[nouvelles_features].select_dtypes("number")
controle = pd.DataFrame({
    "type": df[nouvelles_features].dtypes.astype(str),
    "manquants": df[nouvelles_features].isna().sum(),
    "infinis": np.isinf(numeriques).sum().reindex(nouvelles_features, fill_value=0),
    "nb valeurs": df[nouvelles_features].nunique(),
})
controle

- Aucune valeur infinie (pas de division par zéro).
- Seule `PartUsagePrincipal` a quelques valeurs manquantes (usage principal non renseigné), à imputer lors de la préparation.

In [ ]:
corr_cible = (
    numeriques.corrwith(np.log(df[cible]), method="spearman")
    .sort_values()
)
fig, ax = plt.subplots(figsize=(8, 7))
corr_cible.plot(kind="barh", ax=ax)
ax.set_title("Corrélation de Spearman des nouvelles features avec log(consommation)")
plt.show()

Ce premier aperçu ne remplace pas l'entraînement d'un modèle :
- `LogSurfaceTotale` (≈ 0,74) reste de loin la plus liée à la consommation ; `SurfaceParBatiment` (≈ 0,69) lui est très proche → redondance probable.
- `BuildingAge` et `DistanceCentreKm` sont **négativement** liées (≈ -0,3) : les bâtiments récents et centraux sont en moyenne plus grands.
- Les indicateurs d'usages et de sources d'énergie ont des corrélations faibles (0,1 à 0,3) **prises isolément** : leur effet joue
  **à surface égale** (un restaurant consomme plus qu'un entrepôt de même taille), ce qu'une corrélation univariée ne montre pas.
La sélection finale (suppression des features redondantes) est faite dans la partie suivante.

| Catégorie | Features créées | Colonnes sources | Fuite ? |
|---|---|---|---|
| Temporalité | `BuildingAge`, `PeriodeConstruction` | `YearBuilt` | Non |
| Structure | `LogSurfaceTotale`, `PartParking`, `SurfaceParEtage`, `SurfaceParBatiment`, `MultiBatiments` | surfaces, étages, nb de bâtiments | Non |
| Usages | `NbUsages`, `MultiUsage`, `PartUsagePrincipal/2/3`, `Usage*` (7 binaires), `FamilleUsage` | liste et surfaces des usages, type principal | Non |
| Sources d'énergie | `UtiliseGaz`, `UtiliseVapeur` | présence (> 0) de gaz / vapeur, **jamais la quantité** | Non |
| Localisation | `DistanceCentreKm` | latitude, longitude | Non |

#### Questions que l'évaluateur peut poser

1. *Pourquoi `UtiliseGaz` n'est-il pas une fuite alors que `NaturalGas(kBtu)` en est une ?*
   → C'est la présence d'un équipement (connue sans relevé), pas la quantité consommée qui compose la cible.
2. *Pourquoi ne pas avoir fait `split(",")` directement ?* → Les libellés entre parenthèses contiennent des virgules.
3. *Pourquoi regrouper les types de bâtiments ?* → Certaines modalités ont trop peu de bâtiments pour apprendre un effet fiable.
4. *Vos regroupements créent-ils une fuite entre train et test ?* → Non : ce sont des règles fixées à l'avance, pas des statistiques calculées sur les données.
5. *Pourquoi `LogSurfaceTotale` ?* → Relation linéaire en log-log avec la consommation ; utile pour les modèles linéaires, neutre pour les arbres.

### Préparation des features pour la modélisation

A réaliser :
* Si ce n'est pas déjà fait, supprimer toutes les colonnes peu pertinentes pour la modélisation.
* Tracer la distribution de la cible pour vous familiariser avec l'ordre de grandeur. En cas d'outliers, mettez en place une démarche pour les supprimer.
* Débarrassez-vous des features redondantes en utilisant une matrice de corrélation.
* Réalisez différents graphiques pour comprendre le lien entre vos features et la target (boxplots, scatterplots, pairplot si votre nombre de features numériques n'est pas très élevé).
*  Séparez votre jeu de données en un Pandas DataFrame X (ensemble de feautures) et Pandas Series y (votre target).
* Si vous avez des features catégorielles, il faut les encoder pour que votre modèle fonctionne.

In [ ]:
# CODE PREPARATION DES FEATURES

### Comparaison de différents modèles supervisés

A réaliser :
* Pour chaque algorithme que vous allez tester, vous devez :
    * Réaliser au préalable une séparation en jeu d'apprentissage et jeu de test via une validation croisée.
    * Si les features quantitatives que vous souhaitez utiliser ont des ordres de grandeur très différents les uns des autres, et que vous utilisez un algorithme de regression qui est sensible à cette différence, alors il faut réaliser un scaling (normalisation) de la donnée au préalable.
    * Entrainer le modèle sur le jeu de Train
    * Prédire la cible sur la donnée de test (nous appelons cette étape, l'inférence).
    * Calculer les métriques de performance R2, MAE et RMSE sur le jeu de train et de test.
    * Interpréter les résultats pour juger de la fiabilité de l'algorithme.
* Vous pouvez choisir par exemple de tester un modèle linéaire, un modèle à base d'arbres et un modèle de type SVM
* Déterminer le modèle le plus performant parmi ceux testés.

In [ ]:
# CODE COMPARAISON DES MODELES

### Optimisation et interprétation du modèle

A réaliser :
* Reprennez le meilleur algorithme que vous avez sécurisé via l'étape précédente, et réalisez une GridSearch de petite taille sur au moins 3 hyperparamètres.
* Si le meilleur modèle fait partie de la famille des modèles à arbres (RandomForest, GradientBoosting) alors utilisez la fonctionnalité feature importance pour identifier les features les plus impactantes sur la performance du modèle. Sinon, utilisez la méthode Permutation Importance de sklearn.

In [ ]:
# CODE OPTIMISATION ET INTERPRETATION DU MODELE